# ZMART target acquisition

Run the cells from top to bottom. Edit the setup values, select the requested LAS X jobs when asked, and inspect the plots before acquiring targets.

## 0 · Setup and connect
Edit the values, move to the run origin, then run.

In [ ]:
import sys
sys.path.insert(0, "workflows/target_acquisition")
from _bootstrap import Path, pipeline

ANALYSIS_REPO = Path("C:/code/smart-analysis")
FOCUS_POINTS = [{"x": 0.0, "y": 0.0}, {"x": 5000.0, "y": 0.0}, {"x": 0.0, "y": 5000.0}]
AF_JOB = None
SIMULATE_IMAGES = False
MOCK_IMAGE_SOURCE = "skimage_human_mitosis"

zmart_controller = pipeline.connect("leica")
zmart_controller.set_origin()
OUTPUT_ROOT = pipeline.get_root(zmart_controller)
OUTPUT_ROOT

## 3a · Overview job

Select the low-magnification overview job in LAS X, then run.

In [ ]:
overview_state = zmart_controller.get_state()
overview_state["observed"]

## 3b · Target job

Select the high-magnification target job in LAS X, then run.

In [ ]:
target_state = zmart_controller.get_state()
target_state["observed"]

## 4 · Initial positions
Read overview tile centres from the controller.

In [ ]:
positions = pipeline.get_positions(zmart_controller)
print(len(positions), "overview positions")

## 5 · Focus

Autofocus at the setup points and inspect the fitted surface.

In [ ]:
measured = pipeline.measure_focus(zmart_controller, FOCUS_POINTS, af_job=AF_JOB)
focus = pipeline.fit_focus_surface(measured)
pipeline.plot_focus_surface(focus, save_path=OUTPUT_ROOT / "focus_surface.png", show=True)
print("focus model:", focus.model)

## 6 · Overview

Capture one overview at each position.

In [ ]:
overview_records = pipeline.run_overview(zmart_controller, positions, state=overview_state, focus=focus)
print(len(overview_records), "overviews captured")
if SIMULATE_IMAGES:
    n = pipeline.hijack_if_simulating(
        overview_records,
        simulate=True,
        image_source=MOCK_IMAGE_SOURCE,
    )
    print(n, "overview images replaced")

## 7 · Discover targets

Segment the overview images and convert detected cells to stage positions.

In [ ]:
engine = pipeline.load_analysis_engine(ANALYSIS_REPO)
overviews = pipeline.overview_inputs_from_records(positions, overview_records, focus=focus)
targets = pipeline.discover_targets(engine, overviews)
print(len(targets), "targets discovered")

## 8 · Acquire targets
Acquire the selected targets.

In [ ]:
target_records = pipeline.acquire_targets(zmart_controller, targets, state=target_state, focus=focus)
print(len(target_records), "targets acquired")
if SIMULATE_IMAGES:
    n = pipeline.hijack_if_simulating(
        target_records,
        simulate=True,
        image_source=MOCK_IMAGE_SOURCE,
    )
    print(n, "target images replaced")

## 9 · Summary
Write the summary and run map.

In [ ]:
summary = pipeline.write_run_report(
    OUTPUT_ROOT,
    positions=positions,
    focus=focus,
    overview_records=overview_records,
    targets=targets,
)
summary

In [ ]:
zmart_controller.disconnect()